# Model_VIX_MarketCrash_v5
### Reconstruction propre — S&P 500 × VIX futures (30 / 90 / 180 jours)

**Source de données de marché unique et exclusive :** le classeur Google Sheets `STRATEGY_ANALYSIS` (`1XKeBA6hcnPLcdPixX5bWBtbDTBWP-MtK7Av8ead0NN0`).

**Deux plages autorisées uniquement :** `VIX HISTORIQUE!A4:B7501` (S&P 500) et `VIX From Futures!A1:D5668` (maturités constantes 30 / 90 / 180 jours).

**Aucun accès à d'autres classeurs, feuilles, API de marché, fichiers CSV externes ou téléchargements de prix.** Les bibliothèques Python sont des dépendances logicielles, pas des sources de données.

**Architecture :** ① Configuration → ② Exploration → ③ Calculs → ④ Modèles → ⑤ Visualisations → ⑥ Exports.

> Les résultats sont calculés exclusivement après lecture des deux plages autorisées. Les exports locaux sont des résultats dérivés, jamais des données d'entrée.


## ① Configuration et chargement — source contrôlée

In [ ]:
# Dépendances : pandas, numpy, matplotlib, scikit-learn, google-api-python-client, google-auth
# Dans Colab, si nécessaire : %pip install -q google-api-python-client google-auth scikit-learn
import io, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, precision_score, recall_score

SPREADSHEET_ID = '1XKeBA6hcnPLcdPixX5bWBtbDTBWP-MtK7Av8ead0NN0'
SPX_TAB = 'VIX HISTORIQUE'
VX_TAB = 'VIX From Futures'
CUTOFF = pd.Timestamp('2026-09-30')
HORIZONS = [5, 10, 20, 40, 60]  # jours de Bourse
DRAWDOWNS = [0.05, 0.10, 0.15]
TRAIN_END = pd.Timestamp('2016-12-31')
VALID_END = pd.Timestamp('2021-12-31')
MIN_TRAIN_POSITIVES = 15
OUTPUT = Path('results')
OUTPUT.mkdir(exist_ok=True)
print('Source:', SPREADSHEET_ID, '| coupure:', CUTOFF.date())

**Authentification Google (lecture seule) :** elle sert exclusivement à lire les deux plages autorisées du classeur `STRATEGY_ANALYSIS`. Aucun accès aux autres sources de marché n'est programmé. Dans Colab, authentification interactive ; dans Jupyter local, identifiants Google ADC préconfigurés.


In [ ]:
try:
    from google.colab import auth as colab_auth
    colab_auth.authenticate_user()
except ImportError:
    pass  # Jupyter local : identifiants Google ADC déjà configurés
import google.auth
from googleapiclient.discovery import build
creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/spreadsheets.readonly'])
sheets_api = build('sheets', 'v4', credentials=creds, cache_discovery=False)

# Liste blanche stricte : aucune autre feuille ni aucun autre classeur autorisé.
ALLOWED_RANGES = {
    ('VIX HISTORIQUE', 'A4:B7501'),
    ('VIX From Futures', 'A1:D5668'),
}

def sheet_values(tab, a1):
    if (tab, a1) not in ALLOWED_RANGES:
        raise PermissionError(f"Source non autorisée : {tab}!{a1}")
    result = sheets_api.spreadsheets().values().get(
        spreadsheetId=SPREADSHEET_ID, range=f"'{tab}'!{a1}",
        valueRenderOption='UNFORMATTED_VALUE', dateTimeRenderOption='SERIAL_NUMBER'
    ).execute()
    return result.get('values', [])

spx_rows = sheet_values(SPX_TAB, 'A4:B7501')
vx_rows = sheet_values(VX_TAB, 'A1:D5668')
print('S&P rows:', len(spx_rows), '| futures rows:', len(vx_rows))
assert len(spx_rows) > 1000 and len(vx_rows) > 1000, 'Données insuffisantes : vérifier les onglets / autorisations.' 

## ② Exploration et contrôle de qualité

In [ ]:
def google_date(value):
    if isinstance(value, (int, float)):
        return pd.Timestamp('1899-12-30') + pd.to_timedelta(float(value), unit='D')
    return pd.to_datetime(value, errors='coerce')

spx = pd.DataFrame([r[:2] + [None]*(2-len(r)) for r in spx_rows[1:]], columns=['date','spx_close'])
spx['date'] = spx['date'].map(google_date).dt.normalize()
spx['spx_close'] = pd.to_numeric(spx['spx_close'], errors='coerce')
spx = spx.loc[spx.date.notna() & spx.spx_close.gt(0) & spx.date.le(CUTOFF)].copy()

vx = pd.DataFrame([r[:4] + [None]*(4-len(r)) for r in vx_rows[1:]], columns=['date','VX_CM30','VX_CM90','VX_CM180'])
vx['date'] = pd.to_datetime(vx['date'], errors='coerce').dt.normalize()
for col in ['VX_CM30','VX_CM90','VX_CM180']:
    vx[col] = pd.to_numeric(vx[col], errors='coerce')
vx = vx.loc[vx.date.notna() & vx.date.le(CUTOFF)].copy()

quality = pd.DataFrame([
    {'serie':'SPX','rows':len(spx),'duplicated_dates':int(spx.date.duplicated().sum()),'first':spx.date.min(),'last':spx.date.max(),'missing':int(spx.spx_close.isna().sum())},
    *[{'serie':c,'rows':len(vx),'duplicated_dates':int(vx.date.duplicated().sum()),'first':vx.date.min(),'last':vx.date.max(),'missing':int(vx[c].isna().sum())} for c in ['VX_CM30','VX_CM90','VX_CM180']]
])
display(quality)
assert not spx.date.duplicated().any() and not vx.date.duplicated().any(), 'Dates dupliquées : ne pas agréger silencieusement.'
assert spx.spx_close.gt(0).all(), 'S&P non positif.'
assert ((vx[['VX_CM30','VX_CM90','VX_CM180']] > 0) | vx[['VX_CM30','VX_CM90','VX_CM180']].isna()).all().all(), 'Future non positif.'
spx = spx.sort_values('date').set_index('date')
vx = vx.sort_values('date').set_index('date')
data = spx.join(vx, how='inner').sort_index()
assert len(data) > 1000
print('Dates communes:',len(data),'|',data.index.min().date(),'→',data.index.max().date())
display(data.head(), data.tail(), data.isna().mean().rename('taux manquant').to_frame())

In [ ]:
fig, ax = plt.subplots(2,1,figsize=(14,8),sharex=True)
ax[0].plot(data.index,data.spx_close,label='S&P 500',lw=1);ax[0].set_title('S&P 500 — observations du classeur');ax[0].legend()
for c in ['VX_CM30','VX_CM90','VX_CM180']:
    ax[1].plot(data.index,data[c],label=c,lw=.8)
ax[1].set_title('VIX futures à maturité constante — valeurs brutes du classeur');ax[1].legend(ncol=3)
plt.tight_layout();plt.show()

## ③ Calculs — variables explicatives et événements futurs

**Convention de cible :** à la date *t*, événement = le S&P 500 atteint au moins −X % par rapport à sa clôture en *t* à un moment des **H prochaines séances**. Ce n'est **pas** un drawdown depuis le plus-haut historique. Le signal est calculé à la clôture de *t* ; aucune variable future n'entre dans les features.

**Données manquantes :** aucun remplissage artificiel de VX_CM30. Les modèles nécessitant CM30 excluent les séances correspondantes. Les pentes sont des ratios, et les niveaux/variations restent disponibles comme contrôles.

In [ ]:
d = data.copy()
d['ret_1d'] = d.spx_close.pct_change()
d['spx_mom_20'] = d.spx_close.pct_change(20)
d['spx_vol_20'] = d.ret_1d.rolling(20, min_periods=20).std() * np.sqrt(252)
d['slope_90_30'] = d.VX_CM90 / d.VX_CM30 - 1
d['slope_180_90'] = d.VX_CM180 / d.VX_CM90 - 1
d['slope_180_30'] = d.VX_CM180 / d.VX_CM30 - 1
for c in ['VX_CM30','VX_CM90','VX_CM180']:
    d[f'{c}_chg_5'] = d[c].diff(5)

# Horizon exact : min des H clôtures futures, sans utiliser la clôture courante.
# La cible est NaN si l'horizon complet n'est pas observable (fin de série).
for h in HORIZONS:
    forward_closes = pd.concat([d.spx_close.shift(-k) for k in range(1,h+1)],axis=1)
    future_min = forward_closes.min(axis=1)
    complete = forward_closes.notna().all(axis=1)
    d[f'future_min_ret_{h}d'] = (future_min/d.spx_close - 1).where(complete)
    for dd in DRAWDOWNS:
        label = f'crash_{int(dd*100)}pct_{h}d'
        d[label] = (d[f'future_min_ret_{h}d'] <= -dd).astype(float).where(complete)

FEATURES = ['slope_90_30','slope_180_90','slope_180_30','VX_CM30','VX_CM90','VX_CM180',
            'VX_CM30_chg_5','VX_CM90_chg_5','VX_CM180_chg_5','spx_mom_20','spx_vol_20']
SLOPES = ['slope_90_30','slope_180_90','slope_180_30']
display(d[SLOPES].describe().T)
print('Disponibilité des pentes :'); display(d[SLOPES].notna().mean().to_frame('fraction'))

## ④ Modélisation — validation chronologique, sans fuite

**Trois périodes fixes :** entraînement jusqu'en 2016, validation 2017–2021, test final 2022–septembre 2026. On purge les **H dernières séances** de chaque bloc avant sa frontière pour que les labels d'entraînement/validation ne dépendent pas des prix de la période suivante. Les paramètres ne sont ajustés que sur l'entraînement. **Le test n'est jamais utilisé pour choisir le modèle.**

Modèles comparés : une régression logistique par pente (interprétable), et une régression logistique multivariée. Les métriques ROC-AUC et PR-AUC sont calculées uniquement lorsque les deux classes sont présentes. Les horizons se chevauchent : ne pas interpréter les observations comme indépendantes.

In [ ]:
def make_splits(df,h):
    train = df.loc[df.index <= TRAIN_END].iloc[:-h].copy()
    valid = df.loc[(df.index > TRAIN_END) & (df.index <= VALID_END)].iloc[:-h].copy()
    test = df.loc[df.index > VALID_END].copy()
    return train, valid, test

def score(y,p):
    if len(np.unique(y)) < 2:
        return {'roc_auc':np.nan,'pr_auc':np.nan,'brier':brier_score_loss(y,p),'base_rate':float(y.mean())}
    return {'roc_auc':roc_auc_score(y,p),'pr_auc':average_precision_score(y,p),
            'brier':brier_score_loss(y,p),'base_rate':float(y.mean())}

specs = {**{f'univariate_{s}':[s] for s in SLOPES},
         'three_slopes':SLOPES,
         'all_futures':['VX_CM30','VX_CM90','VX_CM180','slope_90_30','slope_180_90',
                        'VX_CM30_chg_5','VX_CM90_chg_5','VX_CM180_chg_5'],
         'futures_plus_spx':FEATURES}
records=[]; predictions=[]; models={}
for h in HORIZONS:
    tr0,va0,te0=make_splits(d,h)
    for dd in DRAWDOWNS:
        target=f'crash_{int(dd*100)}pct_{h}d'
        for name,cols in specs.items():
            tr=tr0.dropna(subset=cols+[target]); va=va0.dropna(subset=cols+[target]); te=te0.dropna(subset=cols+[target])
            if len(tr)<100 or tr[target].sum()<MIN_TRAIN_POSITIVES or tr[target].nunique()<2:
                records.append({'horizon':h,'threshold':dd,'model':name,'split':'SKIPPED','n':len(tr),
                                'positives':int(tr[target].sum()),'reason':'Insufficient training events'});continue
            model=Pipeline([('scale',StandardScaler()),('logit',LogisticRegression(C=1,max_iter=3000))])
            model.fit(tr[cols],tr[target].astype(int))
            models[(h,dd,name)]=model
            for split,part in [('train',tr),('validation',va),('test',te)]:
                if part.empty:continue
                y=part[target].astype(int).to_numpy();p=model.predict_proba(part[cols])[:,1]
                records.append({'horizon':h,'threshold':dd,'model':name,'split':split,
                                'n':len(part),'positives':int(y.sum()),**score(y,p)})
                predictions.append(pd.DataFrame({'date':part.index,'horizon':h,'threshold':dd,
                                                  'model':name,'split':split,'observed':y,'probability':p}))
metrics=pd.DataFrame(records)
predictions=pd.concat(predictions,ignore_index=True) if predictions else pd.DataFrame()
print('Évaluations:',len(metrics),'| prédictions:',len(predictions))
display(metrics.query("split == 'test'").sort_values('roc_auc',ascending=False).head(20))

### Choix du modèle sans regarder le test
Pour chaque couple (horizon, seuil), le meilleur modèle est celui avec la meilleure **ROC-AUC de validation** ; le score test est présenté seulement après ce choix. Un modèle « choisi » peut avoir un test médiocre : c'est précisément l'intérêt du protocole.

In [ ]:
valid = metrics.loc[metrics.split.eq('validation') & metrics.roc_auc.notna()].copy()
if valid.empty:
    selected = pd.DataFrame()
    print('Pas assez de données de validation pour sélectionner un modèle.')
else:
    winners = valid.sort_values(['horizon','threshold','roc_auc'],ascending=[True,True,False]).drop_duplicates(['horizon','threshold'])
    test_scores = metrics.loc[metrics.split.eq('test')]
    selected = winners[['horizon','threshold','model','roc_auc','pr_auc','base_rate']].rename(columns={'roc_auc':'validation_auc','pr_auc':'validation_pr_auc','base_rate':'validation_base_rate'}).merge(
        test_scores[['horizon','threshold','model','roc_auc','pr_auc','brier','base_rate','n','positives']].rename(columns={'roc_auc':'test_auc','pr_auc':'test_pr_auc','base_rate':'test_base_rate'}),
        on=['horizon','threshold','model'],how='left')
    display(selected.sort_values(['horizon','threshold']))

## ⑤ Visualisations — lecture des signaux et de leur performance

In [ ]:
fig,ax=plt.subplots(3,1,figsize=(15,10),sharex=True)
for a,c in zip(ax,SLOPES):
    a.plot(d.index,d[c],lw=.7,label=c)
    a.axhline(0,color='black',lw=.7,ls='--')
    a.set_ylabel('Pente');a.legend(loc='upper right')
ax[0].set_title('Pentes des futures VIX — uniquement données STRATEGY_ANALYSIS')
plt.tight_layout();plt.show()

In [ ]:
# Exemple interprétable : probabilité de correction -10 % sur 20 séances, test uniquement
example = predictions.loc[(predictions.horizon==20)&(predictions.threshold==0.10)&(predictions.split=='test')].copy() if not predictions.empty else pd.DataFrame()
if not example.empty:
    chosen = selected.loc[(selected.horizon==20)&(selected.threshold==0.10)] if not selected.empty else pd.DataFrame()
    name = chosen.iloc[0]['model'] if not chosen.empty else example.model.iloc[0]
    p = example.loc[example.model==name].sort_values('date')
    fig,ax=plt.subplots(figsize=(14,4))
    ax.plot(p.date,p.probability,label=f'Probabilité estimée : {name}')
    ax.scatter(p.loc[p.observed.eq(1),'date'],p.loc[p.observed.eq(1),'probability'],s=8,label='Événement futur observé',alpha=.6)
    ax.set_ylim(0,1);ax.set_title('Test hors échantillon : correction ≥10 % sous 20 séances');ax.legend();plt.tight_layout();plt.show()
else:
    print('Aucune prédiction disponible pour cet exemple.')

In [ ]:
if not selected.empty:
    heat = selected.pivot(index='threshold',columns='horizon',values='test_auc')
    fig,ax=plt.subplots(figsize=(9,4));im=ax.imshow(heat.to_numpy(),aspect='auto',vmin=0.4,vmax=0.8,cmap='viridis')
    ax.set_xticks(range(len(heat.columns)),heat.columns);ax.set_yticks(range(len(heat.index)),[f'{x:.0%}' for x in heat.index]);
    ax.set_xlabel('Horizon (séances)');ax.set_ylabel('Correction');ax.set_title('ROC-AUC hors échantillon — modèles choisis sur validation')
    for i in range(len(heat.index)):
        for j in range(len(heat.columns)):
            v=heat.iloc[i,j]
            ax.text(j,i,f'{v:.2f}' if pd.notna(v) else 'n/a',ha='center',va='center',color='white')
    fig.colorbar(im,ax=ax,label='ROC-AUC');plt.tight_layout();plt.show()

## ⑥ Exports — données, métriques, prédictions, traçabilité

In [ ]:
d.to_csv(OUTPUT/'v5_dataset_features_targets.csv',index_label='date')
quality.to_csv(OUTPUT/'v5_quality_checks.csv',index=False)
metrics.to_csv(OUTPUT/'v5_model_metrics.csv',index=False)
selected.to_csv(OUTPUT/'v5_selected_models.csv',index=False)
predictions.to_csv(OUTPUT/'v5_predictions.csv',index=False)
manifest = {'model':'Model_VIX_MarketCrash_v5','spreadsheet_id':SPREADSHEET_ID,
            'source_tabs':{'spx':SPX_TAB,'vix_futures':VX_TAB},'data_cutoff':str(CUTOFF.date()),
            'train_end':str(TRAIN_END.date()),'validation_end':str(VALID_END.date()),
            'horizons_trading_days':HORIZONS,'drawdown_thresholds':DRAWDOWNS,
            'target_definition':'min(SPX[t+1:t+H])/SPX[t]-1 <= -threshold',
            'purge':'last H observations of training and validation removed',
            'rows_merged':len(data),'rows_spx':len(spx),'rows_vix':len(vx),
            'external_market_data_used':False}
(OUTPUT/'v5_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Exports créés :')
for p in sorted(OUTPUT.iterdir()):print('•',p)
# Option Colab : compresser les exports pour téléchargement
import zipfile
with zipfile.ZipFile(OUTPUT/'Model_VIX_MarketCrash_v5_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT.iterdir():z.write(p,arcname=p.name)
print('Archive :', OUTPUT/'Model_VIX_MarketCrash_v5_results.zip')

'---
### Contrat de données et limites méthodologiques
- **Source unique** : `STRATEGY_ANALYSIS` ; plages explicitement autorisées et contrôlées dans `sheet_values`.
- Aucune lecture de `VIXFuturesRaw` : les données interpolées déjà validées dans `VIX From Futures` sont utilisées directement.
- Aucune API de données de marché, aucun CSV de données externes, aucune lecture de fichiers de prix locaux.
- **Pas de fuite temporelle** dans les variables explicatives ; purge aux frontières des échantillons.
- **Sélection du modèle** sur la validation uniquement ; **test final** séparé.
- Les événements futurs sur horizons glissants sont autocorrélés : étape suivante, bootstrap par blocs.
- Le modèle ne simule pas une stratégie négociable (coûts, slippage, exécution exclus).
- L'absence de CM30 certaines dates est conservée, jamais extrapolée.
- Les exports locaux sont des résultats calculés, pas des sources d'entrée.
